In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.stattools import adfuller, kpss
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.statespace.sarimax import SARIMAX
from sklearn.metrics import mean_absolute_error, mean_squared_error

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.titlesize"] = 13
plt.rcParams["axes.titleweight"] = "bold"

pd.set_option("display.max_columns", None)


## 1. Data Understanding & Preparation

In [ ]:
df = pd.read_csv(r"Dataset 4_ monthly-beer-production-in-austr.csv")
print(df.shape)
df.head()


In [ ]:
df.info()
print("\nMissing values per column:")
print(df.isna().sum())

In [ ]:
df["Month"] = pd.to_datetime(df["Month"], format="%Y-%m")
df = df.sort_values("Month").set_index("Month")
df.index.duplicated().sum()


NO NA and no Duplicates

In [ ]:
y = df["Monthly beer production"]

print(y.describe())

q1, q3 = y.quantile([0.25, 0.75])
iqr = q3 - q1
lower, upper = q1 - 1.5 * iqr, q3 + 1.5 * iqr
outliers = y[(y < lower) | (y > upper)]
print(f"\nIQR: [{lower:.1f}, {upper:.1f}]")
print(f"Outliers: {len(outliers)}")


## 2. Exploratory Time Series Analysis


In [ ]:
fig, ax = plt.subplots()
ax.plot(y.index, y.values, color="#2c6e91", linewidth=1)
ax.set_title("Monthly Beer Production in Australia (1956–1995)")
ax.set_xlabel("Year")
ax.set_ylabel("Production")
plt.tight_layout()
plt.show()


### Rolling mean and rolling std (12-month window) to visualize local trend & local variability


In [ ]:
roll_mean = y.rolling(window=12, center=True).mean()
roll_std = y.rolling(window=12, center=True).std()

fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=True)
axes[0].plot(y.index, y.values, color="lightgray", label="Original", linewidth=1)
axes[0].plot(roll_mean.index, roll_mean.values, color="#c0392b", label="12-month rolling mean")
axes[0].set_title("Series with 12-Month Rolling Mean (Trend)")
axes[0].legend()

axes[1].plot(roll_std.index, roll_std.values, color="#8e44ad")
axes[1].set_title("12-Month Rolling Standard Deviation")
axes[1].set_xlabel("Year")
plt.tight_layout()
plt.show()


In [ ]:
fig, ax = plt.subplots(1 , figsize=(15, 5))

sns.boxplot(x=y.index.month, y=y.values, ax=ax, color="#a6c8e0")
ax.set_title("Distribution of Production by Calendar Month")
ax.set_xlabel("Month")
ax.set_ylabel("Production")

October , November and December have more production than the rest of months so its Seasonality in these months

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(y, lags=36, ax=axes[0])
axes[0].set_title("ACF — Original Series")
plot_pacf(y, lags=36, method="ywm", ax=axes[1])
axes[1].set_title("PACF — Original Series")
plt.tight_layout()
plt.show()


### ACF not = 0 but there is a strong relation between x and xt-1 
### NON Stationary We must do Differincing 

# 3. Smoothing


In [ ]:
#sma -- Simple Moving Average and ewma -- Exponential Weighted Moving Average
sma_12 = y.rolling(window=12, center=True).mean()
ewma = y.ewm(span=12, adjust=False).mean()

fig, ax = plt.subplots()
ax.plot(y.index, y.values, color="lightgray", linewidth=1, label="Original")
ax.plot(sma_12.index, sma_12.values, color="#c0392b", linewidth=2, label="SMA (window=12, centered)")
ax.plot(ewma.index, ewma.values, color="#27ae60", linewidth=1.5, label="EWMA (span=12)")
ax.set_title("Smoothing: SMA vs. EWMA")
ax.set_xlabel("Year")
ax.set_ylabel("Production")
ax.legend()
plt.tight_layout()
plt.show()


- The **12-month centered SMA** removes essentially all of the seasonal
- The **EWMA** (span=12) is much less aggressive about removing seasonality


# 4. Time Series Decomposition

compare **additive** and **multiplicative** on (period = 12)


In [ ]:
decomp_add = seasonal_decompose(y, model="additive", period=12)
decomp_mul = seasonal_decompose(y, model="multiplicative", period=12)

fig = decomp_add.plot()
fig.set_size_inches(14, 7)
fig.suptitle("Additive Decomposition", fontweight="bold")
plt.tight_layout()
plt.show()

fig = decomp_mul.plot()
fig.set_size_inches(14, 7)
fig.suptitle("Multiplicative Decomposition", fontweight="bold")
plt.tight_layout()
plt.show()


In [ ]:
resid_add = decomp_add.resid.dropna()
resid_mul_abs = ((decomp_mul.resid - 1) * decomp_mul.trend).dropna()

print(f"Additive remainder std       (absolute units): {resid_add.std():.2f}")
print(f"Multiplicative remainder std (converted to absolute units): {resid_mul_abs.std():.2f}")


### We proceed with the **multiplicative decomposition** as the primary one for interpretation, while keeping in mind that a log-additive treatment would be equivalent and is what we will lean on when building the SARIMA model in Section 6.


## 5. Check White Noise

In [ ]:
resid = decomp_mul.resid.dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].plot(resid.index, resid.values, color="#8e44ad", linewidth=0.9)
axes[0].axhline(1.0, color="black", linestyle="--", linewidth=1)
axes[0].set_title("Multiplicative Remainder Over Time")

plot_acf(resid, lags=36, ax=axes[1])
axes[1].set_title("ACF of the Remainder")
plt.tight_layout()
plt.show()


In [ ]:
lb = acorr_ljungbox(resid, lags=[12, 24, 36], return_df=True)
lb


## Ljung NUll Hypothesis Test Failed , showing that p-value almost = 0 and ACF also not equal to zero in some cases so its **not White Noise** and just Decomposition isnt enough so we will go for the SARIMA Model 

## 6. SARIMA Modeling

Since the decomposition remainder is not white noise, we model the series directly with a **SARIMA** model, working on the **log-transformed series** 


In [ ]:
log_y = np.log(y)

fig, ax = plt.subplots()
ax.plot(log_y.index, log_y.values, color="#2c6e91")
ax.set_title("Log-Transformed Series")
plt.tight_layout()
plt.show()


In [ ]:
def adf_report(series, label):
    result = adfuller(series.dropna(), autolag="AIC")
    print(f"--- ADF test: {label} ---")
    print(f"ADF statistic: {result[0]:.3f}")
    print(f"p-value:       {result[1]:.4f}")
    print("Critical values:", {k: round(v, 3) for k, v in result[4].items()})
    verdict = "stationary (reject H0)" if result[1] < 0.05 else "non-stationary (fail to reject H0)"
    print("Verdict:", verdict)
    print()

adf_report(log_y, "log(y) — level")
adf_report(log_y.diff().dropna(), "log(y), first difference (d=1)")
adf_report(log_y.diff(12).dropna(), "log(y), seasonal difference (D=1, period=12)")
adf_report(log_y.diff().diff(12).dropna(), "log(y), d=1 and D=1 combined")


**Differencing decision**
- The **level** log-series is non-stationary (slowly decaying ACF, trend + seasonality both present).
- **Seasonal differencing alone (D=1, no regular differencing)** already removes most of the non-stationarity and gives the *lowest* variance among the options above — regular differencing on top of it (d=1 **and** D=1) reduces variance further only marginally and pushes the lag-1 ACF sharply negative, a classic **over-differencing** symptom.
- We therefore favor **d = 0 or 1 (compare both), D = 1, seasonal period = 12** as the starting point for the SARIMA search, confirmed formally by the ADF results you get when running the cell.


In [ ]:
d1_12 = log_y.diff(12).dropna()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
plot_acf(d1_12, lags=36, ax=axes[0])
axes[0].set_title("ACF — Seasonally Differenced log(y), D=1")
plot_pacf(d1_12, lags=36, method="ywm", ax=axes[1])
axes[1].set_title("PACF — Seasonally Differenced log(y), D=1")
plt.tight_layout()
plt.show()


**Candidate models**

We fit a small grid of SARIMA candidates motivated by the analysis above and compare them by AIC/BIC:

| Model | Motivation |
|---|---|
| SARIMA(0,1,1)(0,1,1)[12] | Classic "airline model" — simplest model consistent with the ACF/PACF pattern |
| SARIMA(1,1,1)(0,1,1)[12] | Adds a non-seasonal AR term to capture the residual short-lag structure |
| SARIMA(2,1,1)(1,1,1)[12] | More flexible: adds seasonal AR and extra non-seasonal AR term |
| SARIMA(1,0,1)(0,1,1)[12] | Tests whether regular differencing (d=1) is actually necessary, or whether d=0 with an AR/MA term suffices |


In [ ]:
orders = [
    ((0, 1, 1), (0, 1, 1, 12), "SARIMA(0,1,1)(0,1,1)[12] — airline model"),
    ((1, 1, 1), (0, 1, 1, 12), "SARIMA(1,1,1)(0,1,1)[12]"),
    ((2, 1, 1), (1, 1, 1, 12), "SARIMA(2,1,1)(1,1,1)[12]"),
    ((1, 0, 1), (0, 1, 1, 12), "SARIMA(1,0,1)(0,1,1)[12] — no regular differencing"),
]

results = []
fitted_models = {}
for order, seasonal_order, label in orders:
    model = SARIMAX(
        log_y, order=order, seasonal_order=seasonal_order,
        enforce_stationarity=False, enforce_invertibility=False,
    )
    fit = model.fit(disp=False)
    fitted_models[label] = fit
    results.append({"model": label, "AIC": fit.aic, "BIC": fit.bic})
    print(f"{label:45s}  AIC={fit.aic:10.2f}   BIC={fit.bic:10.2f}")

results_df = pd.DataFrame(results).sort_values("AIC")
results_df


**Model selection**

Pick the model with the lowest AIC


In [ ]:
best_label = results_df.iloc[0]["model"]
best_order, best_seasonal_order = [
    (order, seasonal_order) for order, seasonal_order, label in orders if label == best_label
][0]

best_model = fitted_models[best_label]
print("Selected model:", best_label)
print(best_model.summary())


## 7. Model Diagnostics

We check whether the residuals of the **selected** SARIMA model are consistent with white noise: no remaining autocorrelation, roughly constant variance, and approximately normal.


In [ ]:
best_model.plot_diagnostics(figsize=(12, 8))
plt.tight_layout()
plt.show()


In [ ]:
resid_model = best_model.resid.dropna()
lb_model = acorr_ljungbox(resid_model, lags=[12, 24, 36], return_df=True)
lb_model


Everything is normal and its white noise

## 8. Forecasting

We split the data chronologically, holding out the **last 24 months** as a test and the rest is train


In [ ]:
h = 24 
train, test = log_y.iloc[:-h], log_y.iloc[-h:]

print(f"Train: {train.index.min().date()} to {train.index.max().date()}  (n={len(train)})")
print(f"Test:  {test.index.min().date()} to {test.index.max().date()}  (n={len(test)})")


In [ ]:
train_model = SARIMAX(
    train, order=best_order, seasonal_order=best_seasonal_order,
    enforce_stationarity=False, enforce_invertibility=False,
).fit(disp=False)

forecast_res = train_model.get_forecast(steps=h)
forecast_mean_log = forecast_res.predicted_mean
conf_int_log = forecast_res.conf_int(alpha=0.05)

forecast_mean = np.exp(forecast_mean_log)
conf_int = np.exp(conf_int_log)
actual_test = np.exp(test)


In [ ]:
fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(y.index, y.values, color="lightgray", label="Full history")
ax.plot(train.index, np.exp(train.values), color="#2c6e91", label="Train")
ax.plot(actual_test.index, actual_test.values, color="black", label="Actual (test)")
ax.plot(forecast_mean.index, forecast_mean.values, color="#c0392b", label="Forecast")
ax.fill_between(
    forecast_mean.index, conf_int.iloc[:, 0], conf_int.iloc[:, 1],
    color="#c0392b", alpha=0.2, label="95% CI",
)
ax.set_xlim(y.index[-72], y.index[-1])
ax.set_title(f"Forecast vs. Actual — {best_label}")
ax.set_ylabel("Production (megalitres)")
ax.legend()
plt.tight_layout()
plt.show()


### Confident = 0.95 and its close enough to the actual values 

## 9. Model Evaluation & Comparison

We evaluate out-of-sample forecast accuracy with **MAE**, **RMSE**, and **MAPE**, and compare this against a naive seasonal baseline (repeat the value from 12 months ago) to make sure the SARIMA model is actually adding value.


In [ ]:
def evaluate(actual, predicted, label):
    mae = mean_absolute_error(actual, predicted)
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mape = np.mean(np.abs((actual - predicted) / actual)) * 100
    print(f"{label:30s}  MAE={mae:7.2f}   RMSE={rmse:7.2f}   MAPE={mape:6.2f}%")
    return {"model": label, "MAE": mae, "RMSE": rmse, "MAPE": mape}

seasonal_naive = y.shift(12).iloc[-h:]

eval_results = []
eval_results.append(evaluate(actual_test.values, forecast_mean.values, best_label))
eval_results.append(evaluate(actual_test.values, seasonal_naive.values, "Seasonal naive (lag-12)"))

eval_df = pd.DataFrame(eval_results)
eval_df


Too Close ! THE Model is GOOD TO GO for the discussion 

In [ ]:
other_eval = []
for order, seasonal_order, label in orders:
    if label == best_label:
        continue
    m = SARIMAX(
        train, order=order, seasonal_order=seasonal_order,
        enforce_stationarity=False, enforce_invertibility=False,
    ).fit(disp=False)
    fc = np.exp(m.get_forecast(steps=h).predicted_mean)
    other_eval.append(evaluate(actual_test.values, fc.values, label))

full_comparison = pd.concat([eval_df, pd.DataFrame(other_eval)], ignore_index=True)
full_comparison.sort_values("RMSE")
